# 09 — Spatial autocorrelation (Moran's I): SASP genes vs matched control genes, and relation to spatial gain
Descriptive (R3.4): does the spatial gain of SASP reflect that SASP genes are unusually spatially smooth? Moran's I uses the model's own spot graph (binary weights, self-loops removed) on all spots of each TEST/HD slide; counts are normalised CP10k → log1p and asserted against the package.
Part 1 regenerates the Moran bank and panel summaries and compares with `revision_spatial_autocorr_2026-10-21/p7m_*`. Part 2 relates the Moran difference to the spatial gain using the stored gene-level trained results (`revision_sasp_specificity_2026-10-20`) and compares with `revision_spatial_autocorr_m2_2026-10-21/p7m2_*`. No training. Needs raw `.h5ad` files (several minutes).

In [ ]:
# --- Setup: mount Drive, unpack the repository, install dependencies ---
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, os
BASE = '/content/drive/MyDrive/New-SASP-idea'                 # data/checkpoints live here (not in the repo)
REPO_ZIP = BASE + '/code_release_export_2026-10-23/spatial-sasp-gat.zip'   # or: git clone <repo URL> /content/spatial-sasp-gat
import shutil, hashlib
if os.path.isdir('/content/spatial-sasp-gat'):
    shutil.rmtree('/content/spatial-sasp-gat')          # never reuse an older unpacked copy: always run the zip that is on Drive
subprocess.run(['unzip', '-q', '-o', REPO_ZIP, '-d', '/content'], check=True)
print('repository zip sha256[:16]:', hashlib.sha256(open(REPO_ZIP, 'rb').read()).hexdigest()[:16], '| size', os.path.getsize(REPO_ZIP))
if 'sasp_gat' in sys.modules: print('WARNING: sasp_gat was already imported in this runtime - reset the runtime so the new code is used')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'], check=True)   # reads HEST .h5ad; scikit-learn / networkx are preinstalled on Colab (do NOT reinstall: it can swap pandas under a running kernel)
# data_tables exported in ENV-2/ENV-3 (copied only if missing in the repo)
import shutil
for f in ['sasp54_genes_canonical_order.csv', 'ppi_string_v12_edges.csv', 'slide_partition.csv']:
    dst = '/content/spatial-sasp-gat/data_tables/' + f
    if not os.path.exists(dst):
        shutil.copy(BASE + '/code_release_export_2026-10-23/' + f, dst)
sys.path.insert(0, '/content/spatial-sasp-gat/src')
import torch, torch_geometric
print('torch', torch.__version__, '| torch_geometric', torch_geometric.__version__,
      '| cuda', torch.cuda.is_available())

In [ ]:
import hashlib, json, pandas as pd, numpy as np, torch
from pathlib import Path
from sasp_gat.config import *
from sasp_gat.data import load_package, split_samples, sha256_file
from sasp_gat.ppi import ppi_edge_index_from_networkx
from sasp_gat.availability import build_availability, effective_mask, score_mask
from sasp_gat.evaluation import load_model, eval_both
from sasp_gat.training import evaluate_multimask_av

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPO = Path('/content/spatial-sasp-gat'); T = REPO / 'data_tables'
PKL = Path(BASE) / 'MODEL_READY_DATASET_v1_2026-09-26.pkl'
pkg, pyg_dataset, gene_order = load_package(PKL)
train_s, val_s, test_s, hd_s = split_samples(pyg_dataset)
AVAIL, LOGGED = build_availability(pyg_dataset.keys(), gene_order, T / 'availability_unmeasured.csv')
ppi_ei = ppi_edge_index_from_networkx(pkg['ppi_graph_networkx'], gene_order, device)
CKDIR = REPO / 'checkpoints' if (REPO / 'checkpoints' / 'full_avail_seed42.pt').exists() else Path(BASE) / 'revision_checkpoints_2026-10-18'
print('checkpoint dir:', CKDIR)
import torch_geometric, scipy; print('versions: torch', torch.__version__, '| pyg', torch_geometric.__version__, '| numpy', np.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__)
import warnings; warnings.filterwarnings('ignore')
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'])   # only needed to read HEST .h5ad coordinates
from sasp_gat.training import vkey
from sasp_gat.baselines import (train_gene_means, legacy_gene_means, eval_baseline, build_neighbour_edges,
                                eval_neighbour_methods)
ABL_DIR = Path(BASE) / 'revision_checkpoints_2026-10-18'
TAB_DIR = Path(BASE) / 'revision_tables_2026-10-18'
BL_DIR  = Path(BASE) / 'revision_spatial_baselines_2026-10-18'
VARIANTS = ['expression-only', 'expression+spatial', 'expression+PPI']
ms = lambda s: f'{s.mean():.4f} ± {s.std(ddof=1):.4f}'
import warnings; warnings.filterwarnings('ignore')
from sasp_gat.verify import compare_csv, report_compare
MR_DIR = Path(BASE) / 'revision_masking_robustness_2026-10-19'
OUT_DIR = Path(BASE) / 'repro_outputs_2026-10-24'; OUT_DIR.mkdir(exist_ok=True)
H5_ROOT = Path(BASE) / 'hest_expression'; H5_DIR = H5_ROOT / 'st' if (H5_ROOT / 'st').exists() else H5_ROOT

from sasp_gat import panels as pn, moran as mo
SPEC = Path(BASE) / 'revision_sasp_specificity_2026-10-20'; M1 = Path(BASE) / 'revision_spatial_autocorr_2026-10-21'; M2 = Path(BASE) / 'revision_spatial_autocorr_m2_2026-10-21'
rd = lambda p: pd.read_csv(p, float_precision='round_trip')
GENES = list(gene_order); DS = pyg_dataset; SPLIT = {s: DS[s].split for s in DS}
S_TE = sorted(s for s in DS if SPLIT[s] == 'test'); S_HD = sorted(s for s in DS if SPLIT[s] == 'external_hd'); SL = S_TE + S_HD
COH = {**{s: 'TEST' for s in S_TE}, **{s: 'HD' for s in S_HD}}
assert len(S_TE) == 13 and len(S_HD) == 12
P = rd(SPEC / 'panels_matched_v1.csv')
assert set(P.rule) == {'md', 'mdv'} and all(g.sasp_gene.tolist() == GENES and g.control_gene.nunique() == 54 and not (set(g.control_gene) & set(GENES)) for _, g in P.groupby(['rule', 'panel']))
files = pn.find_h5(H5_ROOT); assert all(len(files[s]) == 1 for s in SL)


In [ ]:
# Part 1: Moran bank + panel summaries
import time; t0 = time.time()
G, GST = mo.moran_bank(files, DS, SL, COH, P, GENES); print('bank %.0f s' % (time.time() - t0), G.shape)
PS, SD, SC = mo.panel_summaries(G, P, GENES, SL, COH)
out1 = {'p7m_moran_gene_slide_v1.csv': G, 'p7m_panel_slide_summary_v1.csv': PS, 'p7m_sasp_vs_panels_slide_v1.csv': SD, 'p7m_sasp_vs_panels_cohort_v1.csv': SC, 'p7m_graph_stats_v1.csv': GST}
for n, d in out1.items(): d.to_csv(OUT_DIR / n, index=False)
report_compare({n: compare_csv(d, M1 / n, rtol=1e-6, atol=1e-9) for n, d in out1.items()})
print(SC.round(4).to_string(index=False))

In [ ]:
# Part 2: Moran difference vs spatial gain (uses the regenerated bank G and the stored gene-level trained tables)
MO = G.copy()
sp_df, ex_df = rd(SPEC / 'trained_gene_level_v1.csv'), rd(SPEC / 'exprop_gene_level_v1.csv')
ps_h3 = rd(SPEC / 'spatial_contribution_panel_stats_v1.csv'); PAN = rd(SPEC / 'panels_matched_v1.csv')
PL, PSUM, GL, GC = mo.moran_vs_spatial_gain(MO, sp_df, ex_df, ps_h3, PAN)
out2 = {'p7m2_panel_level_v1.csv': PL, 'p7m2_panel_summary_v1.csv': PSUM, 'p7m2_gene_level_v1.csv': GL, 'p7m2_gene_corr_v1.csv': GC}
for n, d in out2.items(): d.to_csv(OUT_DIR / n, index=False)
report_compare({n: compare_csv(d, M2 / n, rtol=1e-6, atol=1e-9) for n, d in out2.items()})
print(PSUM.round(4).to_string(index=False)); print(GC.round(4).to_string(index=False))